# Access control models (RBAC, DAC, MAC)

# Coding problem, 5.2.a: RBAC / DAC / MAC access decision

**Skill:** Mitigate Risk · **Maps to:** 5.2.A · **Language:** Python 3 (pytest)

## Problem
Build the access-decision function a policy engine would run for a fictional file system that
supports all three access control models. Given a request (subject, object, action, and the
object's model), decide whether the request is ALLOWed or DENYed.

## Rules by model
- **`"dac"`**, allowed if the subject is the object's `owner`, OR the subject appears in the
  object's `grants` list for that action (e.g., `grants: {"read": ["bob"], "write": []}`).
- **`"mac"`**, allowed only if the subject's clearance level is `>=` the object's classification
  level. Levels (low to high): `"public"` < `"confidential"` < `"secret"` < `"top_secret"`.
- **`"rbac"`**, allowed if any of the subject's `roles` is present in the object's
  `allowed_roles` list for the requested action (e.g.,
  `allowed_roles: {"read": ["nurse","admin"], "write": ["admin"]}`).

If the object's `model` is not one of `"dac"`, `"mac"`, `"rbac"`, deny with reason `"unknown_model"`.

## Function contract
Implement `solve(request: dict) -> dict` in the code editor (see the Reference section below).

## Input
`request`: a dict with keys `"subject"` (dict) and `"object"` (dict) and `"action"` (str).
See the Reference section below for the exact shape per model.

## Output
`{"allowed": bool, "reason": str}` where reason is one of
`"owner"`, `"granted"`, `"dac_denied"`, `"clearance_sufficient"`, `"clearance_insufficient"`,
`"role_match"`, `"rbac_denied"`, `"unknown_model"`.

## Examples
```
DAC owner:
  {"subject":{"name":"alice"},"object":{"model":"dac","owner":"alice","grants":{}},"action":"read"}
  -> {"allowed": True, "reason": "owner"}

MAC insufficient clearance:
  {"subject":{"clearance":"confidential"},"object":{"model":"mac","classification":"secret"},"action":"read"}
  -> {"allowed": False, "reason": "clearance_insufficient"}

RBAC role match:
  {"subject":{"roles":["nurse"]},"object":{"model":"rbac","allowed_roles":{"read":["nurse","admin"]}},"action":"read"}
  -> {"allowed": True, "reason": "role_match"}
```

## Constraints
- Deterministic, offline, standard library only.
- Must pass all cases in the tests (visible + hidden).

## Reference
Any implementation that satisfies this interface can be swapped in (reference, student, or
composite) and must pass the tests unchanged.

### Entry point
```python
def solve(request: dict) -> dict
```

### Input (`request`)
`{"subject": dict, "object": dict, "action": str}`

- DAC object: `{"model": "dac", "owner": str, "grants": {action: [subject_name...]}}`;
  subject: `{"name": str}`.
- MAC object: `{"model": "mac", "classification": str}`; subject: `{"clearance": str}`.
  Levels low->high: `"public"`, `"confidential"`, `"secret"`, `"top_secret"`.
- RBAC object: `{"model": "rbac", "allowed_roles": {action: [role...]}}`;
  subject: `{"roles": [role...]}`.

### Output (`result`)
`{"allowed": bool, "reason": str}` where `reason` is one of `"owner"`, `"granted"`,
`"dac_denied"`, `"clearance_sufficient"`, `"clearance_insufficient"`, `"role_match"`,
`"rbac_denied"`, `"unknown_model"`.

### Guarantees / rules
- Pure function, deterministic, no I/O, no network. Standard library only.
- DAC: owner check takes priority over grants list.
- MAC: allowed when subject clearance level >= object classification level (equal is allowed).
- RBAC: allowed if any of the subject's roles appears in the object's allowed_roles for the action.
- Any `object.model` outside `{"dac","mac","rbac"}` -> `{"allowed": False, "reason": "unknown_model"}`.

### Acceptance
Your code passes when every test below shows a checkmark (some are hidden until you run them).


In [ ]:
# Your implementation
"""Starter code for 5.2.a. Student completes the TODO.
This file intentionally FAILS the tests until implemented."""
from typing import Dict


def solve(request: Dict[str, object]) -> Dict[str, object]:
    """Decide ALLOW/DENY for a DAC, MAC, or RBAC access request.

    Args:
        request: {"subject": dict, "object": dict, "action": str}
    Returns:
        {"allowed": bool, "reason": str} per the rules in the problem statement above / the Reference section below.
    """
    # TODO: branch on request["object"]["model"] ("dac" | "mac" | "rbac") and apply
    #       the corresponding rule from the problem statement above. Unknown model -> "unknown_model".
    raise NotImplementedError("implement solve() — see the problem statement above")


## Self-check tests
Run the cell below to check your `solve` implementation above.

In [ ]:
import __main__ as impl  # the functions defined above

# ---- visible cases -------------------------------------------------------
def test_dac_owner():
    req = {"subject": {"name": "alice"}, "object": {"model": "dac", "owner": "alice", "grants": {}}, "action": "read"}
    assert impl.solve(req) == {"allowed": True, "reason": "owner"}


def test_dac_granted():
    req = {"subject": {"name": "bob"}, "object": {"model": "dac", "owner": "alice", "grants": {"read": ["bob"]}}, "action": "read"}
    assert impl.solve(req) == {"allowed": True, "reason": "granted"}


def test_dac_denied():
    req = {"subject": {"name": "eve"}, "object": {"model": "dac", "owner": "alice", "grants": {"read": ["bob"]}}, "action": "read"}
    assert impl.solve(req) == {"allowed": False, "reason": "dac_denied"}


def test_mac_sufficient():
    req = {"subject": {"clearance": "secret"}, "object": {"model": "mac", "classification": "confidential"}, "action": "read"}
    assert impl.solve(req) == {"allowed": True, "reason": "clearance_sufficient"}


def test_mac_insufficient():
    req = {"subject": {"clearance": "confidential"}, "object": {"model": "mac", "classification": "secret"}, "action": "read"}
    assert impl.solve(req) == {"allowed": False, "reason": "clearance_insufficient"}


def test_rbac_role_match():
    req = {"subject": {"roles": ["nurse"]}, "object": {"model": "rbac", "allowed_roles": {"read": ["nurse", "admin"]}}, "action": "read"}
    assert impl.solve(req) == {"allowed": True, "reason": "role_match"}


def test_rbac_denied():
    req = {"subject": {"roles": ["intern"]}, "object": {"model": "rbac", "allowed_roles": {"read": ["nurse", "admin"]}}, "action": "read"}
    assert impl.solve(req) == {"allowed": False, "reason": "rbac_denied"}


# ---- hidden / edge cases -------------------------------------------------
def test_unknown_model():
    req = {"subject": {}, "object": {"model": "abac"}, "action": "read"}
    assert impl.solve(req) == {"allowed": False, "reason": "unknown_model"}


def test_mac_equal_level_allowed():
    req = {"subject": {"clearance": "top_secret"}, "object": {"model": "mac", "classification": "top_secret"}, "action": "read"}
    assert impl.solve(req) == {"allowed": True, "reason": "clearance_sufficient"}


def test_dac_owner_priority_over_grants():
    req = {"subject": {"name": "alice"}, "object": {"model": "dac", "owner": "alice", "grants": {"read": []}}, "action": "read"}
    assert impl.solve(req) == {"allowed": True, "reason": "owner"}


def test_rbac_multiple_roles_one_matches():
    req = {"subject": {"roles": ["intern", "admin"]}, "object": {"model": "rbac", "allowed_roles": {"write": ["admin"]}}, "action": "write"}
    assert impl.solve(req) == {"allowed": True, "reason": "role_match"}


# ---- run every test above ------------------------------------------------
_results = []
for _name, _fn in list(globals().items()):
    if _name.startswith("test_") and callable(_fn):
        try:
            _fn()
            _results.append((_name, None))
        except Exception as _e:
            _results.append((_name, _e))
for _name, _e in _results:
    print(("PASS  " if _e is None else "FAIL  ") + _name + ("" if _e is None else f": {_e!r}"))
print(f"{sum(e is None for _, e in _results)}/{len(_results)} tests passed")
